# M4S3 - HPC and Machine Learning
## Training is not only an ML problem - it is a systems problem

In this notebook we will measure how data loading, batch size, precision, memory and parallelism affect machine-learning training.

**Predict → run → observe → explain**

Run the cells in order. Some later cells check whether PyTorch or TensorFlow is available in the current Jupyter kernel.


## 0 - Check the environment
**Predict:** CPU, memory, storage or GPU - what will limit this run?


In [ ]:
import os, sys, shutil, subprocess, time, importlib.util
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np
from IPython.display import display, HTML

def sh(cmd):
    p=subprocess.run(cmd,shell=True,text=True,capture_output=True)
    return (p.stdout+p.stderr).strip()

print("host:", sh("hostname"))
print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("CPU cores:", os.cpu_count())
print("Slurm:", bool(shutil.which("sbatch")))
print("nvidia-smi:", shutil.which("nvidia-smi"))

if shutil.which("nvidia-smi"):
    print(sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader | head -4"))

cmd='bash -lc \'if type module >/dev/null 2>&1; then module -t avail 2>&1 | grep -Ei "pytorch|tensorflow|cuda|cudnn|nccl" | head -20; else echo "module command not visible"; fi\''
print("\nRelevant modules:\n"+sh(cmd))

def show_bars(title, items, unit="", digits=1):
    vals=[float(v) for _,v in items]
    m=max(vals) if vals else 1.0
    html=f"<div style='font-family:Arial,sans-serif;max-width:820px'><h4>{title}</h4>"
    for label,val in items:
        width=0 if m==0 else 100*float(val)/m
        value=f"{float(val):,.{digits}f}{unit}"
        html += (
            "<div style='display:grid;grid-template-columns:150px 1fr 170px;gap:10px;"
            "align-items:center;margin:5px 0'>"
            f"<div>{label}</div>"
            "<div style='background:#e8edf3;height:18px;border-radius:2px'>"
            f"<div style='width:{width:.1f}%;background:#55758f;height:18px;border-radius:2px'></div></div>"
            f"<div style='text-align:right'>{value}</div></div>"
        )
    display(HTML(html+"</div>"))


def run_cluster_command(module_candidates, command):
    """Run a command after loading the first cluster module that works."""
    bootstrap = r"""
if ! type module >/dev/null 2>&1; then
  for f in /etc/profile.d/lmod.sh /etc/profile.d/modules.sh /usr/share/lmod/lmod/init/bash; do
    if [ -r "$f" ]; then source "$f"; break; fi
  done
fi
"""
    for module_name in module_candidates:
        shell = bootstrap + f"""
module purge >/dev/null 2>&1 || true
if ! module load {module_name} >/dev/null 2>&1; then
  exit 97
fi
echo "module: {module_name}"
echo "python: $(command -v python)"
{command}
"""
        p = subprocess.run(["bash","-lc",shell], text=True, capture_output=True)
        output=(p.stdout+p.stderr).strip()
        if p.returncode==0:
            print(output)
            return {"ok":True,"module":module_name,"output":output}
    print("The requested framework module could not be run from this Jupyter session.")
    return {"ok":False}


## 1 - A tiny ML workload
We use a small softmax classifier. The model is intentionally simple: today we care about **data movement, batching, memory and throughput**, not network architecture.


In [ ]:
rng=np.random.default_rng(42); N,D,C=12000,96,5
X=rng.normal(size=(N,D)).astype(np.float32); wt=rng.normal(size=(D,C)).astype(np.float32)
y=np.argmax(X@wt+0.7*rng.normal(size=(N,C)),axis=1).astype(np.int64)
cut=9600; Xtr,Xv=X[:cut],X[cut:]; ytr,yv=y[:cut],y[cut:]
def softmax(z):
    z=z-z.max(1,keepdims=True); e=np.exp(z); return e/e.sum(1,keepdims=True)
def acc(X,y,W,b): return float(np.mean(np.argmax(X@W+b,1)==y))
def grad(X,y,W,b):
    p=softmax(X@W+b); n=len(X); loss=-np.log(p[np.arange(n),y]+1e-12).mean()
    p[np.arange(n),y]-=1; p/=n; return float(loss),X.T@p,p.sum(0)
def train(bs=128,epochs=4,lr=.18,seed=0):
    rg=np.random.default_rng(seed); W=np.zeros((D,C),np.float32); b=np.zeros(C,np.float32); updates=0; t=time.perf_counter()
    for _ in range(epochs):
        ids=rg.permutation(len(Xtr))
        for s in range(0,len(Xtr),bs):
            q=ids[s:s+bs]; _,gW,gb=grad(Xtr[q],ytr[q],W,b); W-=lr*gW; b-=lr*gb; updates+=1
    return W,b,time.perf_counter()-t,updates
W,b,t,u=train()
print(f"samples={len(Xtr):,}, features={D}, data={Xtr.nbytes/2**20:.2f} MiB")
print(f"time={t:.3f}s, updates={u}, throughput={len(Xtr)*4/t:,.0f} samples/s, val_acc={acc(Xv,yv,W,b):.3f}")


## 2 - [DEMO] Optimized libraries
**Predict:** how much faster is NumPy matrix multiplication than nested Python loops?


In [ ]:
A=rng.normal(size=(1200,256)); v=rng.normal(size=256)
def py_mv(A,v):
    out=np.empty(len(A))
    for i,row in enumerate(A):
        s=0.
        for j,x in enumerate(row): s+=x*v[j]
        out[i]=s
    return out
t=time.perf_counter(); a=py_mv(A,v); tp=time.perf_counter()-t
t=time.perf_counter(); 
for _ in range(30): b=A@v
tn=(time.perf_counter()-t)/30
print(f"Python loops: {tp:.4f}s | NumPy: {tn:.6f}s | speedup: {tp/tn:,.0f}x | same result: {np.allclose(a,b)}")


**Lesson:** PyTorch/TensorFlow Python code often calls highly optimized compiled kernels underneath.


## 3 - [DEMO] Preprocessing can starve the accelerator


In [ ]:
raw=rng.integers(0,256,size=(900,96),dtype=np.uint8)
def slow(a):
    z=np.empty(a.shape,np.float32)
    for i in range(len(a)):
        for j in range(a.shape[1]): z[i,j]=(float(a[i,j])-127.5)/127.5
    return z
def fast(a): return (a.astype(np.float32)-127.5)/127.5
t=time.perf_counter(); z1=slow(raw); ts=time.perf_counter()-t
t=time.perf_counter(); z2=fast(raw); tf=time.perf_counter()-t
print(f"slow={len(raw)/ts:,.0f} samples/s | vectorized={len(raw)/tf:,.0f} samples/s | speedup={ts/tf:.1f}x")
for prep in (30,12,4):
    compute=12; print(f"prep={prep:2} ms, compute=12 ms -> approx accelerator utilization={100*compute/(prep+compute):.1f}%")


**Fix the feeding pipeline before buying more GPUs.**


## 4 - [DEMO] Batch size: throughput ↔ memory ↔ convergence
Train the same model for the same 4 epochs.


In [ ]:
rows=[]
for bs in (16,64,256,1024):
    W,b,t,u=train(bs=bs,epochs=4,seed=1)
    rows.append((bs,u,t,len(Xtr)*4/t,acc(Xv,yv,W,b),bs*D*4/1024))
print(f"{'batch':>6} {'updates':>8} {'time':>8} {'samples/s':>12} {'val_acc':>8} {'batch_KiB':>10}")
for r in rows: print(f"{r[0]:6d} {r[1]:8d} {r[2]:8.3f} {r[3]:12,.0f} {r[4]:8.3f} {r[5]:10.1f}")


In [ ]:
show_bars(
    "Measured throughput by batch size",
    [(f"batch {r[0]}", r[3]) for r in rows],
    unit=" samples/s",
    digits=0
)
show_bars(
    "Approximate input-batch memory",
    [(f"batch {r[0]}", r[5]) for r in rows],
    unit=" KiB",
    digits=1
)
print("Validation accuracy:", {r[0]: round(r[4],3) for r in rows})
print("Question: which batch is best if throughput and validation accuracy both matter?")


Discuss:
- larger batches usually improve hardware utilization;
- memory use grows;
- updates per epoch fall;
- convergence can change;
- the batch with the highest throughput is not necessarily the batch with the best validation result.

**Batch size is both an ML parameter and a systems parameter.**


## 5 - [DEMO] Prefetch hides I/O behind compute


In [ ]:
def load(i): time.sleep(.018); return i
def compute(x): time.sleep(.030)
n=12; t=time.perf_counter()
for i in range(n): compute(load(i))
seq=time.perf_counter()-t
t=time.perf_counter()
with ThreadPoolExecutor(max_workers=1) as ex:
    f=ex.submit(load,0)
    for i in range(n):
        x=f.result()
        if i+1<n: f=ex.submit(load,i+1)
        compute(x)
pre=time.perf_counter()-t
print(f"sequential={seq:.3f}s | prefetched={pre:.3f}s | speedup={seq/pre:.2f}x")


Prefetching does not make I/O or compute individually faster; it improves **pipeline utilization** by overlapping them.


## 6 - [DEMO] Precision changes memory and numerical behaviour


In [ ]:
base=rng.normal(size=(2048,512))
for name,dtype in (("FP64",np.float64),("FP32",np.float32),("FP16",np.float16)):
    a=base.astype(dtype); print(f"{name}: {a.nbytes/2**20:.2f} MiB")
A=base[:512,:256]; B=base[:256,:256]; ref=A@B
for name,dtype in (("FP32",np.float32),("FP16",np.float16)):
    a=A.astype(dtype); b=B.astype(dtype); t=time.perf_counter(); out=a@b; dt=time.perf_counter()-t
    err=np.linalg.norm(out.astype(np.float64)-ref)/np.linalg.norm(ref)
    print(f"{name}: time={dt:.4f}s, relative_difference={err:.3e}")


In [ ]:
precision_memory=[("FP64",8.0),("FP32",4.0),("FP16",2.0)]
show_bars("Memory footprint for the same array", precision_memory, unit=" MiB", digits=1)
print("Important: lower precision reduces memory, but speed depends on hardware support.")


> On CPUs FP16 may not be faster. GPUs can have specialized tensor hardware, so **hardware support matters**.


## 7 - [DEMO] How much memory does training state need?

A model needs more memory for training than for storing its parameters alone.

For a simplified FP32 model trained with Adam:

- parameters: **4 bytes / parameter**
- gradients: **4 bytes / parameter**
- Adam first moment: **4 bytes / parameter**
- Adam second moment: **4 bytes / parameter**

Simplified training-state estimate:

**about 16 bytes per parameter**

This excludes activations, temporary buffers, framework overhead and other copies.

**Predict:** could a 7-billion-parameter model train in this simplified form on one 24 GB GPU?


In [ ]:
models=[
    ("1 million",1_000_000),
    ("100 million",100_000_000),
    ("1 billion",1_000_000_000),
    ("7 billion",7_000_000_000),
]
bytes_per_parameter=16
print(f"{'model size':>14} {'training state':>18}")
memory_rows=[]
for label,nparams in models:
    gb=nparams*bytes_per_parameter/1e9
    memory_rows.append((label,gb))
    print(f"{label:>14} {gb:15.3f} GB")
show_bars("Simplified FP32 + Adam training-state memory", memory_rows, unit=" GB", digits=1)
print("\nA 7B model needs about 112 GB for this simplified training state alone.")
print("Activations and temporary tensors would require additional memory.")


**Key idea:** model size can become an HPC memory problem before compute becomes the main problem.

Mixed precision, sharding and distributed strategies can reduce or distribute parts of this footprint. The distributed strategies belong to the next session.


## 8 - [DEMO] Gradient accumulation: a larger effective batch without a larger physical batch

Suppose GPU memory only allows a physical batch of **64**.

Instead of updating after every batch:

**64 examples → backward → update**

we can accumulate gradients:

**64 + 64 + 64 + 64 → backward four times → one update**

Effective batch size:

**64 × 4 = 256**

This helps with memory limits, but it does **not** provide the same compute speedup as four GPUs working in parallel.


In [ ]:
physical_batch=64
accumulation_steps=4
effective_batch=physical_batch*accumulation_steps
print("physical batch:", physical_batch)
print("accumulation steps:", accumulation_steps)
print("effective batch:", effective_batch)
print("Key point: memory is similar to batch 64, but the optimizer updates every 256 examples.")


**Question:** Is gradient accumulation the same as distributed training?

No. Gradient accumulation changes when the model is updated and helps create a larger effective batch. It does not add more GPUs or run the computation in parallel.


## 9 - [DEMO] Checkpointing is resilience + I/O


In [ ]:
out=Path("m4s3_outputs"); out.mkdir(exist_ok=True); f=out/"checkpoint.npz"
W,b,_,_=train(bs=128,epochs=2,seed=3); before=acc(Xv,yv,W,b)
t=time.perf_counter(); np.savez(f,W=W,b=b,epoch=np.array([2])); save=time.perf_counter()-t
del W,b; t=time.perf_counter(); s=np.load(f); load=time.perf_counter()-t
after=acc(Xv,yv,s["W"],s["b"])
print(f"file={f} | size={f.stat().st_size/1024:.1f} KiB | save={save*1e3:.2f} ms | load={load*1e3:.2f} ms")
print(f"accuracy before={before:.4f}, after reload={after:.4f}, recovered={np.isclose(before,after)}")


At scale, checkpoint frequency trades **recovery time** against **storage/I/O overhead**.

Real training jobs may run for hours or days, so losing all progress after a failure can be very expensive.


## 10 - [REAL FRAMEWORK] Train a neural network with PyTorch

This example runs **real PyTorch training** using the cluster module system.

The model performs a small classification task and reports:

- the device used;
- model size;
- training loss by epoch;
- validation accuracy;
- examples processed per second.

If a GPU is allocated to this job, PyTorch uses it. Otherwise the same training runs on CPU.


In [ ]:
pytorch_script = Path("m4s3_outputs/pytorch_train_real.py")
pytorch_script.parent.mkdir(exist_ok=True)

pytorch_script.write_text(r'''
import json
import time
from pathlib import Path
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(7)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N, D, C = 12000, 64, 4

X = torch.randn(N, D)
teacher = torch.randn(D, C)
noise = 0.15 * torch.randn(N, C)
y = (X @ teacher + noise).argmax(dim=1)

split = 10000
train_loader = DataLoader(
    TensorDataset(X[:split], y[:split]),
    batch_size=256,
    shuffle=True,
    num_workers=0,
)

model = nn.Sequential(
    nn.Linear(D, 128),
    nn.ReLU(),
    nn.Linear(128, C),
).to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

def accuracy(x, target):
    model.eval()
    with torch.no_grad():
        pred = model(x.to(device)).argmax(dim=1).cpu()
    return float((pred == target).float().mean())

initial_acc = accuracy(X[split:], y[split:])
losses = []
seen = 0
t0 = time.perf_counter()

for epoch in range(1, 6):
    model.train()
    total_loss = 0.0
    total_examples = 0
    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad(set_to_none=True)
        prediction = model(xb)
        loss = loss_fn(prediction, yb)
        loss.backward()
        optimizer.step()

        total_loss += float(loss) * xb.shape[0]
        total_examples += xb.shape[0]
        seen += xb.shape[0]

    epoch_loss = total_loss / total_examples
    losses.append(epoch_loss)
    print(f"epoch {epoch}: loss={epoch_loss:.4f}")

if device.type == "cuda":
    torch.cuda.synchronize()

elapsed = time.perf_counter() - t0
final_acc = accuracy(X[split:], y[split:])
parameters = sum(p.numel() for p in model.parameters())

print("PyTorch:", torch.__version__)
print("device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print("parameters:", parameters)
print(f"initial validation accuracy: {initial_acc:.3f}")
print(f"final validation accuracy:   {final_acc:.3f}")
print(f"training throughput: {seen/elapsed:,.0f} examples/s")

Path("m4s3_outputs/pytorch_metrics.json").write_text(json.dumps({
    "device": str(device),
    "parameters": parameters,
    "initial_accuracy": initial_acc,
    "final_accuracy": final_acc,
    "losses": losses,
    "throughput_examples_s": seen/elapsed
}))
''')

pt_result = run_cluster_command(
    ["pytorch/2.8.0", "PyTorch/2.1.2-foss-2023a"],
    f'python "{pytorch_script}"'
)

metrics_file = Path("m4s3_outputs/pytorch_metrics.json")
if pt_result["ok"] and metrics_file.exists():
    import json
    m=json.loads(metrics_file.read_text())
    show_bars(
        "PyTorch validation accuracy",
        [("before training", 100*m["initial_accuracy"]), ("after training", 100*m["final_accuracy"])],
        unit="%",
        digits=1,
    )
    show_bars(
        "PyTorch loss by epoch",
        [(f"epoch {i+1}",v) for i,v in enumerate(m["losses"])],
        digits=3,
    )


### What is real in this example?

The notebook has actually:

1. created tensors;
2. built a neural network;
3. executed forward passes;
4. computed a loss;
5. used automatic differentiation for backpropagation;
6. updated model parameters with Adam;
7. measured validation accuracy and throughput.

**Question:** If the GPU is waiting for input data, will a faster GPU solve the bottleneck?


## 11 - [DEMO] Why gradient averaging works

Take one batch of **512 examples**.

First compute its gradient as one batch.

Then split the same batch into four equal shards:

**128 + 128 + 128 + 128**

Each shard produces a local gradient. Averaging the four local gradients should reproduce the full-batch gradient.

This is the numerical idea behind synchronous data parallelism.


In [ ]:
ids=rng.choice(len(Xtr),512,replace=False); xb,yb=Xtr[ids],ytr[ids]
W=np.zeros((D,C),np.float32); b=np.zeros(C,np.float32)
_,gf,bf=grad(xb,yb,W,b); gs=[]; bs=[]
for q in np.array_split(np.arange(512),4):
    _,g,bb=grad(xb[q],yb[q],W,b); gs.append(g); bs.append(bb)
ga=np.mean(gs,0); ba=np.mean(bs,0)
print("4 workers x 128 samples")
print("max gradient difference:",float(np.max(np.abs(gf-ga))))
print("max bias-gradient difference:",float(np.max(np.abs(bf-ba))))


The averaged gradient is almost identical to the gradient computed from the full batch.

This is the key idea behind synchronous data parallel training:

**split data → compute locally → reduce gradients → update the same model**

With real GPUs, the gradient reduction becomes communication between devices.


## 12 - [REAL DISTRIBUTED ML] PyTorch DistributedDataParallel

Now run **real PyTorch DistributedDataParallel (DDP)** with two processes.

Each process:

1. owns a full model replica;
2. receives a different shard through `DistributedSampler`;
3. computes local gradients;
4. participates in gradient synchronization;
5. applies the same optimizer update.

The demo also executes a real `all_reduce` collective and measures its time.

If two GPUs are available, DDP uses **NCCL + one process per GPU**. Otherwise it uses **Gloo + two CPU processes**, so the distributed mechanism can still be observed.


In [ ]:
ddp_script = Path("m4s3_outputs/pytorch_ddp_real.py")

ddp_script.write_text(r'''
import os
import time
import torch
import torch.distributed as dist
from torch import nn
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, TensorDataset, DistributedSampler

dist.init_process_group(backend="nccl" if torch.cuda.is_available() and torch.cuda.device_count() >= dist.get_world_size() else "gloo")

rank = dist.get_rank()
world = dist.get_world_size()
local_rank = int(os.environ.get("LOCAL_RANK", rank))

use_gpu = dist.get_backend() == "nccl"
device = torch.device(f"cuda:{local_rank}" if use_gpu else "cpu")
if use_gpu:
    torch.cuda.set_device(local_rank)

torch.manual_seed(11)
N, D, C = 8000, 48, 4
X = torch.randn(N, D)
teacher = torch.randn(D, C)
y = (X @ teacher + 0.10 * torch.randn(N, C)).argmax(dim=1)

dataset = TensorDataset(X, y)
sampler = DistributedSampler(dataset, num_replicas=world, rank=rank, shuffle=True)
loader = DataLoader(dataset, batch_size=128, sampler=sampler, num_workers=0)

model = nn.Sequential(
    nn.Linear(D, 96),
    nn.ReLU(),
    nn.Linear(96, C),
).to(device)

ddp = DDP(model, device_ids=[local_rank] if use_gpu else None)
optimizer = torch.optim.SGD(ddp.parameters(), lr=0.12)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(3):
    sampler.set_epoch(epoch)
    total = 0.0
    count = 0
    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad(set_to_none=True)
        loss = loss_fn(ddp(xb), yb)
        loss.backward()
        optimizer.step()

        total += float(loss) * xb.shape[0]
        count += xb.shape[0]

    loss_pair = torch.tensor([total, count], dtype=torch.float64, device=device)
    dist.all_reduce(loss_pair, op=dist.ReduceOp.SUM)
    if rank == 0:
        print(f"epoch {epoch+1}: global loss={loss_pair[0].item()/loss_pair[1].item():.4f}")

# Check that all replicas finished with identical parameters.
checksum = torch.tensor(
    [sum(float(p.detach().float().sum()) for p in ddp.module.parameters())],
    dtype=torch.float64,
    device=device,
)
checks = [torch.zeros_like(checksum) for _ in range(world)]
dist.all_gather(checks, checksum)

# Measure an actual collective operation.
payload = torch.ones(250_000, dtype=torch.float32, device=device)
dist.barrier()
if use_gpu:
    torch.cuda.synchronize()
t0 = time.perf_counter()
for _ in range(10):
    x = payload.clone()
    dist.all_reduce(x, op=dist.ReduceOp.SUM)
if use_gpu:
    torch.cuda.synchronize()
allreduce_ms = (time.perf_counter() - t0) * 1000 / 10

# Evaluate the synchronized model.
with torch.no_grad():
    pred = ddp.module(X.to(device)).argmax(dim=1).cpu()
accuracy = float((pred == y).float().mean())

if rank == 0:
    values = [v.item() for v in checks]
    print("backend:", dist.get_backend())
    print("world size:", world)
    print("device mode:", "GPU" if use_gpu else "CPU")
    print(f"final accuracy: {accuracy:.3f}")
    print(f"replica checksum spread: {max(values)-min(values):.3e}")
    print(f"1 MB all_reduce average: {allreduce_ms:.3f} ms")

dist.destroy_process_group()
''')

ddp_result = run_cluster_command(
    ["pytorch/2.8.0", "PyTorch/2.1.2-foss-2023a"],
    f'python -m torch.distributed.run --standalone --nproc_per_node=2 "{ddp_script}"'
)


### What to look for

A successful run should show:

- **world size = 2**: two real training processes;
- a real DDP backend: **Gloo** on CPU or **NCCL** on GPU;
- training loss changing over epochs;
- a **replica checksum spread near zero**, showing synchronized model replicas;
- measured time for a real **All-Reduce** collective.

This is not a simulation of DDP. It uses PyTorch's distributed runtime.


## 13 - [REAL DISTRIBUTED ML] TensorFlow MirroredStrategy

Now run a second implementation of synchronous data parallelism with **TensorFlow MirroredStrategy**.

The model is replicated across two devices. The global batch is split between replicas, gradients are reduced, and the replicas receive the same update.

If two GPUs are allocated, the demo uses them. Otherwise it creates two logical CPU devices so that TensorFlow still executes the distributed strategy.


In [ ]:
tensorflow_script = Path("m4s3_outputs/tensorflow_mirrored_real.py")

tensorflow_script.write_text(r'''
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import time
import numpy as np
import tensorflow as tf

tf.random.set_seed(13)
np.random.seed(13)

tf.config.threading.set_inter_op_parallelism_threads(2)
tf.config.threading.set_intra_op_parallelism_threads(2)

physical_gpus = tf.config.list_physical_devices("GPU")

if len(physical_gpus) >= 2:
    devices = ["/GPU:0", "/GPU:1"]
    device_mode = "GPU"
else:
    cpus = tf.config.list_physical_devices("CPU")
    if not cpus:
        raise RuntimeError("No CPU device is visible to TensorFlow.")
    tf.config.set_logical_device_configuration(
        cpus[0],
        [tf.config.LogicalDeviceConfiguration(), tf.config.LogicalDeviceConfiguration()]
    )
    logical_cpus = tf.config.list_logical_devices("CPU")
    devices = [logical_cpus[0].name, logical_cpus[1].name]
    device_mode = "logical CPU"

strategy = tf.distribute.MirroredStrategy(devices=devices)

N, D, C = 10000, 40, 4
rng = np.random.default_rng(13)
X = rng.normal(size=(N, D)).astype("float32")
teacher = rng.normal(size=(D, C)).astype("float32")
y = np.argmax(X @ teacher + 0.10 * rng.normal(size=(N, C)), axis=1).astype("int32")

split = 8000

with strategy.scope():
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(D,)),
        tf.keras.layers.Dense(96, activation="relu"),
        tf.keras.layers.Dense(C),
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(0.01),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=["accuracy"],
    )

t0 = time.perf_counter()
history = model.fit(
    X[:split],
    y[:split],
    batch_size=256,
    epochs=4,
    verbose=0,
)
elapsed = time.perf_counter() - t0

loss, accuracy = model.evaluate(X[split:], y[split:], verbose=0)

print("TensorFlow:", tf.__version__)
print("device mode:", device_mode)
print("replicas in sync:", strategy.num_replicas_in_sync)
print("devices:", devices)
for i, value in enumerate(history.history["loss"], start=1):
    print(f"epoch {i}: loss={value:.4f}")
print(f"validation accuracy: {accuracy:.3f}")
print(f"training throughput: {(split*4)/elapsed:,.0f} examples/s")
''')

tf_result = run_cluster_command(
    ["TensorFlow/2.13.0-foss-2023a"],
    f'python "{tensorflow_script}"'
)


### Compare the two frameworks

| Concept | PyTorch | TensorFlow |
|---|---|---|
| Model replicas | DDP | MirroredStrategy |
| Data sharding | DistributedSampler | strategy distributes the batch |
| Gradient synchronization | DDP collectives | strategy collectives |
| Multi-node option | multi-node DDP | MultiWorkerMirroredStrategy |

The APIs differ, but the HPC pattern is the same:

**replicate model → split data → compute locally → reduce gradients → synchronized update**


## 14 - [ACTIVITY] Which distributed strategy fits the problem?

| Problem | Strategy | Example |
|---|---|---|
| Model fits on one GPU, training is slow | **Data parallelism** | image classifier on 4 GPUs |
| One node is not enough | **Multi-worker data parallelism** | 2 nodes x 4 GPUs |
| Training state is too large for one GPU | **State sharding** | PyTorch FSDP |
| Model layers do not fit on one GPU | **Model or pipeline parallelism** | very large transformer |

**Predict before revealing the answer:** Which row describes a 7B-parameter model whose training state exceeds the memory of one accelerator?

### Framework names

**PyTorch:** DDP for data parallelism, FSDP for sharded training state  
**TensorFlow:** MirroredStrategy on one host, MultiWorkerMirroredStrategy across hosts


### Worked example: 4 GPUs with data parallelism

**Global batch = 512**

- GPU 0 gets 128 examples
- GPU 1 gets 128 examples
- GPU 2 gets 128 examples
- GPU 3 gets 128 examples

All four GPUs hold the same model.

After the backward pass:

**4 local gradients → collective reduction → synchronized gradient → same update on every replica**

**Question:** If compute time per GPU becomes 20 ms but gradient synchronization takes 18 ms, would adding more GPUs still scale well?


## 15 - Real multi-GPU and multi-node launch patterns

### PyTorch: one node, four GPUs

```bash
torchrun --standalone --nproc_per_node=4 train_ddp.py
```

### PyTorch: two nodes, four GPUs per node

```bash
torchrun --nnodes=2 --nproc_per_node=4 \
         --node_rank=<0-or-1> \
         --master_addr=<host> --master_port=29500 \
         train_ddp.py
```

This gives **8 training processes**, normally one process per GPU.

### TensorFlow: one host

```python
strategy = tf.distribute.MirroredStrategy()
```

### TensorFlow: several workers

```python
strategy = tf.distribute.MultiWorkerMirroredStrategy()
```

A multi-worker TensorFlow job also needs a cluster description such as `TF_CONFIG`.

These are the commands and APIs used for real jobs. The exact Slurm wrapper depends on the cluster.


## 16 - [REAL CLUSTER] What resources does an ML job need?

Inspect the actual Slurm resources visible from this environment.

A GPU training job usually needs:

- **GPU:** forward and backward computation;
- **CPU cores:** loading, decoding and preprocessing;
- **memory:** batches and worker processes;
- **storage bandwidth:** datasets and checkpoints.

A typical request could include:

```bash
#SBATCH --job-name=ml-demo
#SBATCH --gpus=1
#SBATCH --cpus-per-task=8
#SBATCH --mem=32G
```

**Question:** Why can a GPU job still need many CPU cores?


In [ ]:
if shutil.which("sinfo"):
    print("Visible Slurm partitions and resources:")
    print(sh("sinfo -o '%P  GRES=%G  CPUs=%c  MemMB=%m' | head -15"))
else:
    print("sinfo is not visible in this environment.")


# 17 - Exit ticket

A training job shows:

- GPU utilization: **35%**
- GPU memory: **45%**
- CPU utilization: **100%**
- continuous storage reads

**Would adding more GPUs be your first change? Why or why not?**

<details>
<summary>Check your answer</summary>

Probably not. The evidence suggests that CPU preprocessing or the data pipeline is the main limitation. Extra GPUs may spend more time waiting for input.

</details>

### Remember

**pipeline → bottleneck → evidence → change one factor → measure again**

### Next

Multiple GPUs add compute capacity, but they also add **communication and synchronization**.
